In [1]:
#Cài đặt thuật giải A* tìm đường đi giữa 2 đỉnh bất kỳ trong đồ thị.
import heapq

def a_star_algorithm(graph, start_node, stop_node, heuristics):
    # open_list chứa các đỉnh đã được duyệt nhưng chưa mở rộng hoàn toàn
    # closed_list chứa các đỉnh đã được mở rộng hoàn toàn
    open_list = [(0 + heuristics[start_node], 0, start_node, [start_node])]
    visited = {}

    while open_list:
        (f, g, current_node, path) = heapq.heappop(open_list)

        if current_node == stop_node:
            return path, g

        if current_node in visited and visited[current_node] <= g:
            continue

        visited[current_node] = g

        for (neighbor, weight) in graph.get(current_node, []):
            new_g = g + weight
            new_f = new_g + heuristics.get(neighbor, 0)
            heapq.heappush(open_list, (new_f, new_g, neighbor, path + [neighbor]))

    return None, float('inf')

In [2]:
# Định nghĩa đồ thị dưới dạng danh sách kề: {đỉnh: [(đỉnh kề, trọng số)]}
graph = {
    'A': [('B', 1), ('C', 3)],
    'B': [('D', 2), ('E', 1)],
    'C': [('F', 5)],
    'D': [('G', 3)],
    'E': [('G', 1)],
    'F': [('G', 2)],
    'G': []
}

# Định nghĩa hàm Heuristic (khoảng cách ước lượng đến đích 'G')
heuristics = {
    'A': 6, 'B': 4, 'C': 4, 'D': 2, 'E': 1, 'F': 1, 'G': 0
}

start = 'A'
goal = 'G'
path, cost = a_star_algorithm(graph, start, goal, heuristics)

print(f"Đường đi ngắn nhất từ {start} đến {goal}: {' -> '.join(path)}")
print(f"Tổng chi phí: {cost}")

Đường đi ngắn nhất từ A đến G: A -> B -> E -> G
Tổng chi phí: 3


In [3]:

#Cài đặt bài toán người giao hàng theo thuật giải A*.import heapq
def get_mst_weight(nodes, distance_matrix):
    if not nodes:
        return 0
    nodes = list(nodes)
    unvisited = set(nodes)
    if not unvisited:
        return 0

    start_node = nodes[0]
    unvisited.remove(start_node)
    edges = []
    for n in unvisited:
        heapq.heappush(edges, (distance_matrix[start_node][n], n))

    mst_weight = 0
    while unvisited and edges:
        weight, u = heapq.heappop(edges)
        if u in unvisited:
            unvisited.remove(u)
            mst_weight += weight
            for v in unvisited:
                heapq.heappush(edges, (distance_matrix[u][v], v))
    return mst_weight

def tsp_a_star(distance_matrix):
    n = len(distance_matrix)
    start_node = 0
    all_visited = (1 << n) - 1

    # Priority Queue: (f_score, current_node, visited_mask, current_cost, path)
    pq = [(0, start_node, 1 << start_node, 0, [start_node])]

    while pq:
        f, u, mask, g, path = heapq.heappop(pq)

        if mask == all_visited:
            # Return to start node
            final_cost = g + distance_matrix[u][start_node]
            return path + [start_node], final_cost

        unvisited_nodes = [v for v in range(n) if not (mask & (1 << v))]

        for v in unvisited_nodes:
            new_g = g + distance_matrix[u][v]
            # Heuristic: MST of unvisited nodes + dist to nearest unvisited + unvisited to start
            h = get_mst_weight(unvisited_nodes, distance_matrix)
            new_f = new_g + h

            heapq.heappush(pq, (new_f, v, mask | (1 << v), new_g, path + [v]))

    return None, float('inf')

In [4]:
# Ma trận khoảng cách giữa 4 thành phố (0, 1, 2, 3)
distance_matrix = [
    [0, 10, 15, 20],
    [10, 0, 35, 25],
    [15, 35, 0, 30],
    [20, 25, 30, 0]
]

path, total_cost = tsp_a_star(distance_matrix)

print(f"Đường đi tối ưu: {' -> '.join(map(str, path))}")
print(f"Tổng chi phí nhỏ nhất: {total_cost}")

Đường đi tối ưu: 0 -> 1 -> 3 -> 2 -> 0
Tổng chi phí nhỏ nhất: 80
